# Support Vector Machine Classifier — From Scratch (Linear & Kernel)

_Generated: 2025-10-22T17:51:43.366541Z_

This notebook implements **soft‑margin SVM** from first principles:

- **Linear SVM** optimized in the **primal** via **Pegasos** (stochastic subgradient) and a simple batch subgradient.
- **Kernel SVM** with an **RBF kernel** using a minimal **SMO** (Platt‑style) solver for the **dual**.

We include synthetic datasets (2D), optional CSV upload, visualization of the separating hyperplane and margins, and evaluation metrics.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports, Utilities, and Plot Helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 42
rng = np.random.default_rng(SEED)

def train_test_split(X, y, test_size=0.25, rng=rng):
    n = len(y); idx = np.arange(n); rng.shuffle(idx)
    n_test = int(round(test_size*n))
    te = idx[:n_test]; tr = idx[n_test:]
    return X[tr], X[te], y[tr], y[te]

def standardize(X, mean=None, std=None, eps=1e-12):
    if mean is None: mean = X.mean(axis=0)
    if std is None: std = X.std(axis=0)
    std = np.where(std < eps, 1.0, std)
    return (X - mean)/std, mean, std

def add_intercept(X):
    return np.hstack([np.ones((X.shape[0],1)), X])

def metrics(y_true, y_pred):
    acc = float(np.mean(y_true == y_pred))
    tp = int(np.sum((y_true==1) & (y_pred==1)))
    tn = int(np.sum((y_true==-1) & (y_pred==-1)))
    fp = int(np.sum((y_true==-1) & (y_pred==1)))
    fn = int(np.sum((y_true==1) & (y_pred==-1)))
    prec = tp/(tp+fp+1e-12); rec = tp/(tp+fn+1e-12)
    return acc, prec, rec, tp, fp, tn, fn

def plot_data_and_boundary(ax, X, y, predict_fn=None, title=None):
    ax.scatter(X[y==1,0], X[y==1,1], s=20, label="+1")
    ax.scatter(X[y==-1,0], X[y==-1,1], s=20, label="-1")
    if predict_fn is not None:
        x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
        y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        grid = np.c_[xx.ravel(), yy.ravel()]
        zz = predict_fn(grid).reshape(xx.shape)
        cs = ax.contour(xx, yy, zz, levels=[-1,0,1])
    if title: ax.set_title(title)
    ax.legend(loc="best")

## 2) Data — Synthetic 2D and Optional CSV Upload

In [ ]:
def make_linear(n=400):
    m1 = np.array([0.8, 0.8]); m0 = np.array([-0.8, -0.8])
    X1 = rng.normal(m1, 0.6, size=(n//2, 2))
    X0 = rng.normal(m0, 0.6, size=(n//2, 2))
    X = np.vstack([X1, X0])
    y = np.r_[np.ones(n//2), -np.ones(n//2)]
    idx = rng.permutation(n)
    return X[idx], y[idx]

def make_moons(n=500, noise=0.2):
    t = rng.random(n)*np.pi
    x1 = np.c_[np.cos(t), np.sin(t)] + noise*rng.normal(size=(n,2))*0.3
    x2 = np.c_[1-np.cos(t), 1-np.sin(t)] + noise*rng.normal(size=(n,2))*0.3
    X = np.vstack([x1, x2])
    y = np.r_[np.ones(n), -np.ones(n)]
    idx = rng.permutation(2*n)
    return X[idx], y[idx]

DATASET = "moons"  # "linear" or "moons"

if DATASET == "linear":
    X, y = make_linear(500)
else:
    X, y = make_moons(400, noise=0.5)

# Optional CSV upload (expects numeric features and a label in {0,1} or {-1,1})
USE_CSV = False
LABEL_COL = None
FEATURE_COLS = None

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and USE_CSV:
    print("Upload a CSV file...")
    up = files.upload()
    import io, csv
    name = list(up.keys())[0]
    raw = up[name].decode("utf-8", errors="ignore")
    reader = csv.reader(io.StringIO(raw))
    rows = list(reader)
    header = rows[0]; data = rows[1:]
    def to_num(v):
        try: return float(v)
        except: return np.nan
    M = np.array([[to_num(v) for v in row] for row in data], dtype=float)
    if isinstance(LABEL_COL, str):
        y = M[:, header.index(LABEL_COL)]
    else:
        y = M[:, LABEL_COL if LABEL_COL is not None else -1]
    if FEATURE_COLS is None:
        FEATURE_COLS = [j for j in range(M.shape[1]) if j != (LABEL_COL if isinstance(LABEL_COL, int) else header.index(LABEL_COL))]
    X = M[:, FEATURE_COLS]
    # Map labels to {-1, +1}
    if set(np.unique(y)) == {0.0, 1.0}:
        y = np.where(y>0.5, 1.0, -1.0)
    else:
        y = np.where(y>0, 1.0, -1.0)

print("X shape:", X.shape, "| class balance:", (y==1).mean())

## 3) Preprocess — Standardize and Split

In [ ]:
X_std, mu, sigma = standardize(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_std, y, test_size=0.25, rng=rng)
print("Train:", X_tr.shape, "Test:", X_te.shape)

## 4) Linear SVM (Primal) — Pegasos (Stochastic Subgradient)

We minimize the soft‑margin objective in the primal:
$$\min_w \tfrac{\lambda}{2}\lVert w\rVert^2 + \tfrac{1}{n}\sum_i \max\{0, 1 - y_i w^\top x_i\}.$$
Pegasos update for a sample $(x_i,y_i)$ at iteration $t$ with step $\eta_t = 1/(\lambda t)$:
\[ w \leftarrow (1-\eta_t\lambda)w + \begin{cases} \eta_t y_i x_i, & \text{if } y_i w^\top x_i < 1 \\ 0, & \text{otherwise.}\end{cases} \]

In [ ]:
def pegasos(X, y, lam=1e-2, iters=50_000, seed=0):
    rng_local = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    for t in range(1, iters+1):
        i = int(rng_local.integers(0, n))
        xi, yi = X[i], y[i]
        eta = 1.0/(lam*t)
        margin = yi*np.dot(w, xi)
        w = (1 - eta*lam)*w
        if margin < 1.0:
            w += eta*yi*xi
    return w

def decision_linear(X, w, b):
    return X @ w + b

def fit_linear_svm_pegasos(X, y, lam=1e-2, iters=50_000):
    # Learn w; then choose b to maximize margin center by averaging support-ish points
    w = pegasos(X, y, lam=lam, iters=iters, seed=SEED)
    scores = X@w
    # estimate b by placing decision boundary between closest positive and negative projections
    pos = scores[y==1]; neg = scores[y==-1]
    b = -0.5*(np.min(pos) + np.max(neg))
    return w, b

### 4.1 (Optional) Batch Subgradient for Linear SVM

In [ ]:
def batch_subgrad(X, y, lam=1e-2, steps=2000, step0=0.5):
    n, d = X.shape
    w = np.zeros(d)
    for t in range(1, steps+1):
        margins = y*(X@w)
        mis = margins < 1.0
        grad = lam*w - np.mean((y[mis, None]*X[mis]), axis=0) if np.any(mis) else lam*w
        eta = step0/np.sqrt(t)
        w -= eta*grad
    scores = X@w
    pos = scores[y==1]; neg = scores[y==-1]
    b = -0.5*(np.min(pos) + np.max(neg))
    return w, b

## 5) Kernel SVM (RBF) — Dual with Simplified SMO

We solve the dual problem with box constraints `0 ≤ αᵢ ≤ C` and `∑ αᵢ yᵢ = 0` using a minimal **SMO** routine.
The decision function is `f(x)=∑ αᵢ yᵢ K(xᵢ,x)+b` with RBF kernel `K(x,z)=exp(-γ‖x−z‖²)`.

In [ ]:
def rbf_kernel(X, Z, gamma):
    # returns Gram matrix [len(X), len(Z)]
    X2 = np.sum(X*X, axis=1)[:,None]
    Z2 = np.sum(Z*Z, axis=1)[None,:]
    D2 = X2 + Z2 - 2*X@Z.T
    return np.exp(-gamma*np.maximum(D2, 0.0))

class KernelSVM_SMO:
    def __init__(self, C=1.0, gamma=1.0, tol=1e-3, max_passes=5):
        self.C=C; self.gamma=gamma; self.tol=tol; self.max_passes=max_passes
        self.alphas=None; self.b=0.0; self.sv_idx=None; self.X=None; self.y=None; self.K=None

    def fit(self, X, y):
        n = len(y)
        self.X = X; self.y = y.astype(float)
        self.alphas = np.zeros(n)
        self.b = 0.0
        self.K = rbf_kernel(X, X, self.gamma)
        passes = 0
        while passes < self.max_passes:
            num_changed = 0
            for i in range(n):
                Ei = (self.alphas*self.y)@self.K[:,i] + self.b - self.y[i]
                if ((self.y[i]*Ei < -self.tol and self.alphas[i] < self.C) or
                    (self.y[i]*Ei >  self.tol and self.alphas[i] > 0)):
                    j = i
                    while j == i:
                        j = int(rng.integers(0, n))
                    Ej = (self.alphas*self.y)@self.K[:,j] + self.b - self.y[j]
                    ai, aj = self.alphas[i], self.alphas[j]
                    if self.y[i] != self.y[j]:
                        L = max(0.0, aj - ai)
                        H = min(self.C, self.C + aj - ai)
                    else:
                        L = max(0.0, ai + aj - self.C)
                        H = min(self.C, ai + aj)
                    if L == H: continue
                    eta = 2*self.K[i,j] - self.K[i,i] - self.K[j,j]
                    if eta >= 0: continue
                    new_aj = aj - (self.y[j]*(Ei - Ej))/eta
                    new_aj = np.clip(new_aj, L, H)
                    if abs(new_aj - aj) < 1e-5:
                        continue
                    new_ai = ai + self.y[i]*self.y[j]*(aj - new_aj)
                    b1 = (self.b - Ei
                          - self.y[i]*(new_ai - ai)*self.K[i,i]
                          - self.y[j]*(new_aj - aj)*self.K[i,j])
                    b2 = (self.b - Ej
                          - self.y[i]*(new_ai - ai)*self.K[i,j]
                          - self.y[j]*(new_aj - aj)*self.K[j,j])
                    if 0 < new_ai < self.C:
                        new_b = b1
                    elif 0 < new_aj < self.C:
                        new_b = b2
                    else:
                        new_b = 0.5*(b1+b2)
                    self.alphas[i] = new_ai
                    self.alphas[j] = new_aj
                    self.b = new_b
                    num_changed += 1
            if num_changed == 0:
                passes += 1
            else:
                passes = 0
        self.sv_idx = np.where(self.alphas > 1e-6)[0]
        return self

    def decision(self, Z):
        if self.sv_idx is None or len(self.sv_idx)==0:
            return np.zeros(len(Z))
        Kxz = rbf_kernel(self.X[self.sv_idx], Z, self.gamma)
        return (self.alphas[self.sv_idx]*self.y[self.sv_idx])@Kxz + self.b

    def predict(self, Z):
        return np.where(self.decision(Z) >= 0.0, 1.0, -1.0)

## 6) Train Linear SVMs (Pegasos + Batch Subgradient)

In [ ]:
LAM = 1e-2
w_pg, b_pg = fit_linear_svm_pegasos(X_tr, y_tr, lam=LAM, iters=40_000)
w_bs, b_bs = batch_subgrad(X_tr, y_tr, lam=LAM, steps=2500, step0=0.7)

def dec_lin_pg(Z): return decision_linear(Z, w_pg, b_pg)
def dec_lin_bs(Z): return decision_linear(Z, w_bs, b_bs)

yhat_pg = np.where(dec_lin_pg(X_te) >= 0, 1.0, -1.0)
yhat_bs = np.where(dec_lin_bs(X_te) >= 0, 1.0, -1.0)

def hinge_loss(X, y, w, b):
    margins = y*(X@w + b)
    return float(np.mean(np.maximum(0.0, 1.0 - margins)))

acc_pg, prec_pg, rec_pg, *_ = metrics(y_te, yhat_pg)
acc_bs, prec_bs, rec_bs, *_ = metrics(y_te, yhat_bs)
print(f"Pegasos: acc={acc_pg:.3f} prec={prec_pg:.3f} rec={rec_pg:.3f} hinge={hinge_loss(X_te,y_te,w_pg,b_pg):.3f}")
print(f"Batch  : acc={acc_bs:.3f} prec={prec_bs:.3f} rec={rec_bs:.3f} hinge={hinge_loss(X_te,y_te,w_bs,b_bs):.3f}")

## 7) Visualization — Linear SVM Decision Boundary & Margins

In [ ]:
fig = plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1)
def sign_pg(Z): return dec_lin_pg(Z)
plot_data_and_boundary(ax1, X_tr, y_tr, predict_fn=sign_pg, title="Linear SVM (Pegasos) — Train")

ax2 = plt.subplot(1,2,2)
def sign_bs(Z): return dec_lin_bs(Z)
plot_data_and_boundary(ax2, X_tr, y_tr, predict_fn=sign_bs, title="Linear SVM (Batch) — Train")
plt.tight_layout(); plt.show()

## 8) Train Kernel SVM (RBF) with SMO

In [ ]:
C = 1.5
GAMMA = 1.5 if DATASET=="moons" else 0.8
ksvm = KernelSVM_SMO(C=C, gamma=GAMMA, tol=1e-3, max_passes=8).fit(X_tr, y_tr)
yhat_k = ksvm.predict(X_te)
acc_k, prec_k, rec_k, *_ = metrics(y_te, yhat_k)
print(f"Kernel RBF SVM: acc={acc_k:.3f} prec={prec_k:.3f} rec={rec_k:.3f} #SV={len(ksvm.sv_idx)}")

## 9) Visualization — Kernel SVM Decision Regions

In [ ]:
fig = plt.figure(figsize=(5,4.5))
ax = plt.gca()
def sign_k(Z): return ksvm.decision(Z)
plot_data_and_boundary(ax, X_tr, y_tr, predict_fn=sign_k, title="Kernel SVM (RBF) — Train")
# Mark support vectors
sv = X_tr[ksvm.sv_idx]
ax.scatter(sv[:,0], sv[:,1], s=60, facecolors='none', edgecolors='k')
plt.tight_layout(); plt.show()

## 10) Margin Width (Linear SVM)

In [ ]:
def margin_width(w):
    return 2.0/np.linalg.norm(w) if np.linalg.norm(w)>0 else np.inf
print("Margin width (Pegasos linear):", margin_width(w_pg))

## 11) Artifacts & Download

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)
np.savez("artifacts/svm_run.npz",
         dataset=np.array([DATASET], dtype=object),
         mu=mu, sigma=sigma,
         w_pg=w_pg, b_pg=b_pg,
         w_bs=w_bs, b_bs=b_bs,
         C=np.array([C]), gamma=np.array([GAMMA]),
         ksvm_alphas=ksvm.alphas, ksvm_b=np.array([ksvm.b]), ksvm_sv_idx=ksvm.sv_idx,
         X_te=X_te, y_te=y_te)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 12) Exercises & Extensions

- Add **cross‑validation** to tune `λ` (linear) or `(C, γ)` (RBF).
- Implement **coordinate descent** in the primal with hinge loss.
- Add **Polynomial** and **Sigmoid** kernels.
- Compare against **logistic regression** and **perceptron** on the same data.
- Implement **one‑vs‑rest** multi‑class SVM and measure macro/micro metrics.
